# Teaching a Neural Network to Compose
Build and train a small LSTM that learns melodies and generates new ones.
Run top to bottom on Colab (CPU is fine). Total training time: a few minutes.

**Idea:** music is a sequence of notes. We train the model to predict the *next note* from the previous 32.

## Quick glossary (plain English)
- **Neural network:** a program made of many tiny number-crunching units that learns from examples.
- **MIDI file:** a list of notes, like sheet music for computers.
- **Token:** the ID number we give each different note.
- **Embedding:** turns each note into a list of numbers that describe it.
- **LSTM:** a layer built to remember what came earlier in a sequence.
- **Loss:** a score for how wrong the guesses are (lower is better).
- **Overfitting:** memorising the practice music instead of learning the patterns.
- **Temperature:** a dial for how adventurous the generated music is.

In [ ]:
!pip -q install pretty_midi
import random, glob, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import Audio
random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)

## 1. Data
By default we use a **built-in pentatonic corpus** (short phrases built from repeated motifs), so no downloads are needed.
To train on your own music, upload `.mid` files to a folder and set `MIDI_DIR` (only use music you own or have permission to use).

In [ ]:
MIDI_DIR = None   # e.g. "my_midi/"  (folder of .mid files) ; None = built-in corpus

def builtin_corpus(n_phrases=300, length=64):
    scale = [60, 62, 64, 67, 69, 72, 74, 76, 79]   # C major pentatonic over ~1.5 octaves
    songs = []
    for _ in range(n_phrases):
        motif = [random.choice(scale) for _ in range(4)]
        seq = []
        while len(seq) < length:
            m = motif if random.random() < 0.7 else [random.choice(scale) for _ in range(4)]
            seq += m
        songs.append(seq[:length])
    return songs

def midi_corpus(folder):
    import pretty_midi
    songs = []
    for f in glob.glob(folder + "/*.mid") + glob.glob(folder + "/*.midi"):
        try:
            pm = pretty_midi.PrettyMIDI(f)
            notes = sorted([n for i in pm.instruments if not i.is_drum for n in i.notes], key=lambda n: n.start)
            if len(notes) > 40: songs.append([n.pitch for n in notes])
        except Exception as e:
            print("skip", f, e)
    return songs

songs = midi_corpus(MIDI_DIR) if MIDI_DIR else builtin_corpus()
print(len(songs), "pieces;", sum(map(len, songs)), "notes")

## 2. Turn notes into numbers
Each distinct pitch becomes an integer token. We slice every piece into windows: 32 notes in, 1 note out.

In [ ]:
pitches = sorted({p for s in songs for p in s})
p2i = {p:i for i,p in enumerate(pitches)}; i2p = {i:p for p,i in p2i.items()}
V, SEQ = len(pitches), 32
print("vocabulary size:", V)

X, y = [], []
for s in songs:
    ids = [p2i[p] for p in s]
    for i in range(len(ids) - SEQ):
        X.append(ids[i:i+SEQ]); y.append(ids[i+SEQ])
X = torch.tensor(X); y = torch.tensor(y)
perm = torch.randperm(len(X)); X, y = X[perm], y[perm]
cut = int(0.9 * len(X))
Xtr, ytr, Xva, yva = X[:cut], y[:cut], X[cut:], y[cut:]
print("train windows:", len(Xtr), "| validation windows:", len(Xva))

## 3. The model
Embedding → LSTM → Linear. Roughly 100K parameters.

In [ ]:
class MusicNet(nn.Module):
    def __init__(self, V, emb=64, hid=128):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.lstm = nn.LSTM(emb, hid, batch_first=True)
        self.out = nn.Linear(hid, V)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.out(h[:, -1])          # logits for the next note

model = MusicNet(V).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## 4. Train
Watch the validation loss: if training loss keeps falling while validation loss rises, the model is memorising (overfitting).

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
lossf = nn.CrossEntropyLoss()
EPOCHS, BS = 12, 128
tr_hist, va_hist = [], []

for ep in range(EPOCHS):
    model.train(); idx = torch.randperm(len(Xtr)); tot = 0
    for i in range(0, len(Xtr), BS):
        b = idx[i:i+BS]
        xb, yb = Xtr[b].to(device), ytr[b].to(device)
        opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    model.eval()
    with torch.no_grad():
        va = lossf(model(Xva.to(device)), yva.to(device)).item()
    tr_hist.append(tot / len(Xtr)); va_hist.append(va)
    print(f"epoch {ep+1:2d}  train {tr_hist[-1]:.3f}  val {va:.3f}")

plt.plot(tr_hist, label="train"); plt.plot(va_hist, label="validation")
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.legend(); plt.show()

## 5. Generate music
**Temperature** controls creativity: low = safe and repetitive, high = adventurous and risky.

In [ ]:
@torch.no_grad()
def generate(n_notes=64, temperature=1.0, seed=None):
    model.eval()
    seed = seed or songs[random.randrange(len(songs))][:SEQ]
    ids = [p2i[p] for p in seed[:SEQ]]
    out = list(ids)
    for _ in range(n_notes):
        x = torch.tensor([out[-SEQ:]]).to(device)
        logits = model(x)[0] / temperature
        nxt = torch.multinomial(torch.softmax(logits, -1), 1).item()
        out.append(nxt)
    return [i2p[i] for i in out]

def to_audio(pitches, sr=22050, dur=0.28):
    t = np.linspace(0, dur, int(sr*dur), False)
    env = np.exp(-4*t)                                   # quick decay so notes sound plucked
    wave = np.concatenate([np.sin(2*np.pi*440*2**((p-69)/12)*t)*env for p in pitches])
    return wave.astype(np.float32), sr

melody = generate(64, temperature=1.0)
print(melody[:16], "...")
wave, sr = to_audio(melody)
Audio(wave, rate=sr)

In [ ]:
for T in (0.5, 1.0, 1.5):
    print("Temperature", T)
    w, sr = to_audio(generate(48, temperature=T))
    display(Audio(w, rate=sr))

## 6. Your turn
1. Change `temperature` and compare what you hear.
2. Increase `EPOCHS` or `hid` and watch the validation loss.
3. Set `MIDI_DIR` to a folder of your own MIDI files (benga, gospel, Gengetone loops you made) and retrain.
4. Stretch goal: also predict note *duration*, or replace the LSTM with a small Transformer.

**Responsible use:** only train on music you own or have licensed, and credit artists whose work inspires yours.